In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os 
from pathlib import Path

### create masks

#### create "extract-mask" and visualize rows

In [2]:
dir_path = Path('../test/data')
csv_files_path = file_paths = sorted([str(p) for p in dir_path.rglob("*.csv") if p.is_file()])

In [3]:
for i, file_path in enumerate(file_paths):
    print(f"File {str(i).rjust(2, ' ')}: {file_path}")

File  0: ..\test\data\derivatives_bars_daily_futures.csv_data.csv
File  1: ..\test\data\derivatives_bars_daily_options.csv_data.csv
File  2: ..\test\data\derivatives_bars_daily_options_225.csv_data.csv
File  3: ..\test\data\equities_bars_daily.csv_data.csv
File  4: ..\test\data\equities_earnings-calendar.csv_data.csv
File  5: ..\test\data\equities_investor-types.csv_data.csv
File  6: ..\test\data\equities_master.csv_data.csv
File  7: ..\test\data\fins_details.csv_data.csv
File  8: ..\test\data\fins_dividend.csv_data.csv
File  9: ..\test\data\fins_summary.csv_data.csv
File 10: ..\test\data\indices_bars_daily.csv_data.csv
File 11: ..\test\data\indices_bars_daily_topix.csv_data.csv
File 12: ..\test\data\markets_breakdown.csv_data.csv
File 13: ..\test\data\markets_calendar.csv_data.csv
File 14: ..\test\data\markets_margin-alert.csv_data.csv
File 15: ..\test\data\markets_margin-interest.csv_data.csv
File 16: ..\test\data\markets_short-ratio.csv_data.csv


In [4]:
df_calender = pd.read_csv("markets_calendar.csv")
df_equities_bars_daily = pd.read_csv(csv_files_path[3])

In [5]:
df_equities_bars_daily['date'] = pd.to_datetime(df_equities_bars_daily['date'])
df_calender['Date'] = pd.to_datetime(df_calender['Date'])

In [6]:
df_equities_bars_daily = df_equities_bars_daily.fillna(-1)

In [7]:
df = pd.read_csv(csv_files_path[3])
start_day:str = df[df['date'] == '2008-01-01']['date'].values[0]
end_day:str = df[len(df)-1:]['date'].values[0]
print(f"Start day: {start_day}, End day: {end_day}")

Start day: 2008-01-01, End day: 2026-04-17


In [8]:
from_idx = df_calender[df_calender['Date'] == start_day].index[0]
end_idx  = df_calender[df_calender['Date'] == end_day].index[0]
print(f'from={from_idx}, to={end_idx}, total={end_idx - from_idx + 1}days')

from=0, to=6681, total=6682days


in total 
<span style="color: lightblue; ">6682</span> days, 
<span style="color: lightblue; ">2008-01-01</span> to 
<span style="color: lightblue; ">2026-04-17</span>

In [9]:
plt.figure(figsize=(300, 60))
plt.grid()
plt.plot(df_equities_bars_daily['date'], df_equities_bars_daily['rows'], label='Rows Number', color='blue', lw=10)
plt.xlabel('Date', fontsize=100)
plt.ylabel('Rows Number', fontsize=100)
plt.title('equities_bars_daily of Rows Number Over Time; Raw data', fontsize=300)
plt.legend(fontsize=300)
plt.xticks(rotation=45, fontsize=100)
plt.yticks(fontsize=100)
plt.ylim([-200, 4500])
plt.savefig('../images/raw_data_rows_number_of_equities_bars_daily_over_time.png')
plt.show()

In [10]:
df_equities_bars_daily['date'] = pd.to_datetime(df_equities_bars_daily['date'])
df_calender['Date'] = pd.to_datetime(df_calender['Date'])

In [ ]:
df_extract_by_date = df_equities_bars_daily[
    (df_equities_bars_daily['date'] >= start_day) & 
    (df_equities_bars_daily['date'] <= end_day)
]

df_calender_extract_by_date = df_calender[
    (df_calender['Date'] >= start_day) & 
    (df_calender['Date'] <= end_day)
]

assert len(df_extract_by_date) == len(df_calender_extract_by_date)

In [31]:
extract_date_mask = (df_equities_bars_daily['date'] >= start_day) & (df_equities_bars_daily['date'] <= end_day)

assert len(extract_date_mask) == len(df_equities_bars_daily)

np.save('../masks/extract_date_mask.npy', extract_date_mask)

In [32]:
Holiday_mask = (df_calender_extract_by_date['HolDiv'] == 1) | (df_calender_extract_by_date['HolDiv'] == 2)

assert len(df_extract_by_date) == len(Holiday_mask)

In [33]:
df_extract_by_date.index = Holiday_mask.index
masked_df_extract_by_date = df_extract_by_date[Holiday_mask]

In [34]:
masked_df_extract_by_date

,date,name,cols,rows
3,2008-01-04,equities_bars_daily.csv,-1.0,-1.0
6,2008-01-07,equities_bars_daily.csv,-1.0,-1.0
7,2008-01-08,equities_bars_daily.csv,-1.0,-1.0
8,2008-01-09,equities_bars_daily.csv,-1.0,-1.0
9,2008-01-10,equities_bars_daily.csv,-1.0,-1.0
...,...,...,...,...
6677,2026-04-13,equities_bars_daily.csv,42.0,4449.0
6678,2026-04-14,equities_bars_daily.csv,42.0,4448.0
6679,2026-04-15,equities_bars_daily.csv,42.0,4448.0
6680,2026-04-16,equities_bars_daily.csv,42.0,4448.0


HolDiv（休日区分）が１（営業日）か２（東証半日立会日）の時に分析対象とする

- ０は「非営業日」でありデータが登録されていないため利用しない
- ３は「非営業日(祝日取引あり)」であり，NaNを多く踏むため利用しない

In [35]:
plt.figure(figsize=(300, 60))
plt.grid()
plt.plot(masked_df_extract_by_date['date'], masked_df_extract_by_date['rows'], label='Rows Number', color='blue', lw=10)
plt.xlabel('Date', fontsize=100)
plt.ylabel('Rows Number', fontsize=100)
plt.title('equities_bars_daily of Rows Number Over Time; Limit (Holdiv: [1, 2] Range: [2008-01-01 - 2026-04-17])', fontsize=300)
plt.legend(fontsize=300)
plt.xticks(rotation=45, fontsize=100)
plt.yticks(fontsize=100)
plt.ylim([-200, 4500])
plt.savefig('../images/masked_data_rows_number_of_equities_bars_daily_over_time.png')
plt.show()

In [36]:
cols_and_rows_Nan_mask = ~((df_extract_by_date["cols"] < 0) & (df_extract_by_date["rows"] < 0))
assert len(cols_and_rows_Nan_mask) == len(Holiday_mask)
complete_mask = Holiday_mask & cols_and_rows_Nan_mask

In [37]:
usable_df = df_extract_by_date[complete_mask]

In [38]:
plt.figure(figsize=(300, 60))
plt.grid()
plt.plot(usable_df['date'], usable_df['rows'], label='Rows Number', color='blue', lw=10)
plt.xlabel('Date', fontsize=100)
plt.ylabel('Rows Number', fontsize=100)
plt.title('equities_bars_daily of Rows Number Over Time; Limit (Holdiv: [1, 2] Range: [2008-01-01 - 2026-04-17])', fontsize=300)
plt.legend(fontsize=300)
plt.xticks(rotation=45, fontsize=100)
plt.yticks(fontsize=100)
plt.ylim([-200, 4500])
plt.savefig('../images/usable_data_rows_number_of_equities_bars_daily_over_time.png')
plt.show()

#### completely create "date-mask"
1. HolDiv only use 1 and 2
2. TimeRange only use 2008-01-01 - 2026-04-17 [reference from trade-calender]
3. cols and rows only use non-negative values [data-vendor don't provide data for those days]
4. target-node is stock-node, so use time-range valid-stock-data-range

In [40]:
np.save('../masks/valid_date_mask.npy', complete_mask)

#### create only-date-nparray

In [11]:
extract_date_mask = np.load('../masks/extract_date_mask.npy')
valid_date_mask = np.load('../masks/valid_date_mask.npy')
df_equities_bars_daily = pd.read_csv("../test/data/equities_master.csv_data.csv")

df_equities_bars_daily = df_equities_bars_daily[extract_date_mask][valid_date_mask].reset_index(drop=True)

In [12]:
df_equities_bars_daily

,date,name,cols,rows
0,2008-05-07,equities_master.csv,13,2494
1,2008-05-08,equities_master.csv,13,2494
2,2008-05-09,equities_master.csv,13,2493
3,2008-05-12,equities_master.csv,13,2493
4,2008-05-13,equities_master.csv,13,2493
...,...,...,...,...
4387,2026-04-13,equities_master.csv,13,4449
4388,2026-04-14,equities_master.csv,13,4448
4389,2026-04-15,equities_master.csv,13,4448
4390,2026-04-16,equities_master.csv,13,4448


In [16]:
only_date_nparray = df_equities_bars_daily['date']

In [17]:
np.save('../masks/only_date_nparray.npy', only_date_nparray)